In [1]:
import gym
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
env = gym.make('CartPole-v0')

### Setup R packages

In [2]:
from cxrl.lib.r_to_py import setup_R
setup_R()

[]


### A sample policy for Cartpole

In [3]:
def policy(state):
    x0 = state[0]/2.4
    x2 = state[2]/2.095
    if x0*0.4+x2*0.6>=0:
        return 1
    else:
        return 0

### Infer the global causal graph from episodes

Proposed method elapsed time

In [4]:
from cxrl.lib.xrl import XRL
# names for interpretable representations
feature_names = ['cart_position', 'cart_velocity', 'pole_angle', 'pole_velocity', 'push']
xrl = XRL(env, pi=policy, repr_names=feature_names, verbose=True, repr_integration=False, nepisodes=200, method='expectation')

experience size= 6236
 causes: ['cart_position', 'pole_angle'] -> cart_position
 causes: ['pole_angle'] -> cart_velocity
 causes: ['pole_angle'] -> pole_angle
 causes: ['pole_angle'] -> pole_velocity
 causes: [] -> push
BART SLA elapsed time: 0.06602025032043457 seconds


In [5]:
states, actions, rewards, states_new = xrl.replay_buffer

from cxrl.lib.utils import convert_and_expand
states = convert_and_expand(states).astype(float)
actions = convert_and_expand(actions).astype(float)

### Customized CI test embedded simulations of BART CI test

In [9]:
from causallearn.utils.cit import CIT, CIT_Base, register_ci_test

from sklearn.model_selection import train_test_split
import time
import warnings
warnings.filterwarnings('ignore')


class CustomCIT(CIT_Base):
    total_time = 0.0

    @classmethod
    def reset_timer(cls):
        cls.total_time = 0.0

    @classmethod
    def get_total_time(cls):
        return cls.total_time

    def __init__(self, data, **kwargs):
        super().__init__(data, **kwargs)
        self.method = "custom_test"

        # Important: do not recreate Fisher-Z inside every CI call
        self.fisherz_obj = CIT(self.data, "fisherz")

    def __call__(self, X, Y, S=None):
        if S is None:
            S = []

        X_train, X_test, y_train, y_test = train_test_split(
            self.data,
            self.data[:, Y],
            test_size=0.5
        )

        y_hat_cf = (np.random.rand(500, y_test.shape[0]) - 0.5) * 10

        start_time = time.time()

        # simulation block
        sse_cf = (y_test - y_hat_cf) ** 2
        mse_cf = sse_cf.mean(axis=1)
        _, _ = np.quantile(mse_cf, [0.05, 0.95])
        _ = np.quantile(sse_cf, 0.95)

        elapsed_time = time.time() - start_time
        CustomCIT.total_time += elapsed_time

        # actual CI test
        p_value = self.fisherz_obj(X, Y, S)
        return p_value


# Register the CI test
register_ci_test("custom_test", CustomCIT)

sla_data = np.concatenate((states, actions), axis=1)

### Customized socre function

In [27]:
from causallearn.search.ScoreBased import GES as ges_module
from causallearn.score.LocalScoreFunction import local_score_BIC as original_local_score_BIC

class CustomGESScore:
    total_time = 0.0

    @classmethod
    def reset_timer(cls):
        cls.total_time = 0.0

    @classmethod
    def get_total_time(cls):
        return cls.total_time


    @staticmethod
    def local_score(Data, i, PAi, parameters=None):
        """
        Customized local score function for GES.

        Data: numpy array, shape = (n_samples, n_features)
        i: target variable index
        PAi: list of parent indices for variable i
        parameters: optional score parameters

        Must return a scalar score. Higher score is better in causal-learn GES.
        """

        PAi = list(PAi)

        # simulation block
        _, X_test, _, y_test = train_test_split(
            Data,
            Data[:, i],
            test_size=0.5
        )

        y_hat_cf = (np.random.rand(500, y_test.shape[0]) - 0.5) * 10

        start_time = time.time()

        sse_cf = (y_test - y_hat_cf) ** 2
        mse_cf = sse_cf.mean(axis=1)
        _, _ = np.quantile(mse_cf, [0.05, 0.95])
        _ = np.quantile(sse_cf, 0.95)

        elapsed_time = time.time() - start_time

        CustomGESScore.total_time += elapsed_time

        # actual bic score
        score = original_local_score_BIC(Data, i, PAi, parameters)

        return score


def ges_with_custom_bic_score(X, maxP=None, parameters=None, node_names=None):
    """
    Temporarily replace causal-learn's local_score_BIC with the custom score,
    run GES, then restore the original functions.
    """

    # Save originals
    original_ges_local_score_BIC = ges_module.local_score_BIC

    # Some causal-learn versions internally route BIC through
    # local_score_BIC_from_cov, so save and patch it if it exists.
    has_bic_from_cov = hasattr(ges_module, "local_score_BIC_from_cov")
    if has_bic_from_cov:
        original_ges_local_score_BIC_from_cov = ges_module.local_score_BIC_from_cov

    try:
        # Patch BIC score used inside GES
        ges_module.local_score_BIC = CustomGESScore.local_score

        if has_bic_from_cov:
            ges_module.local_score_BIC_from_cov = CustomGESScore.local_score

        # Run GES. Keep score_func as "local_score_BIC".
        Record = ges_module.ges(
            X,
            score_func="local_score_BIC",
            maxP=maxP,
            parameters=parameters,
            node_names=node_names
        )

    finally:
        # Always restore original functions
        ges_module.local_score_BIC = original_ges_local_score_BIC

        if has_bic_from_cov:
            ges_module.local_score_BIC_from_cov = original_ges_local_score_BIC_from_cov

    return Record

In [29]:
from causallearn.search.ConstraintBased.PC import pc
from causallearn.search.ConstraintBased.FCI import fci
from causallearn.search.ScoreBased.GES import ges

# ---------- PC algorithm ----------
print("Learn causal graph using PC algorithm...")
CustomCIT.reset_timer()

cg = pc(
    sla_data,
    alpha=0.05,
    indep_test="custom_test",
    stable=False
)

pc_ci_time = CustomCIT.get_total_time()
print(f"* PC algorithm SLA elapsed CI-test time: {pc_ci_time:.4f} seconds\n")


# ---------- FCI algorithm ----------
print("Learn causal graph using FCI algorithm...")
CustomCIT.reset_timer()

g, edges = fci(
    sla_data,
    independence_test_method="custom_test",
    alpha=0.05,
    verbose=False
)

fci_ci_time = CustomCIT.get_total_time()
print(f"* FCI algorithm SLA elapsed CI-test time: {fci_ci_time:.4f} seconds\n")



# ---------- GES algorithm ----------
print("Learn causal graph using GES algorithm...")
CustomGESScore.reset_timer()

Record = ges_with_custom_bic_score(
    sla_data,
    maxP=None
)

print(f"* GES algorithm SLA elapsed score-computation time: "
      f"{CustomGESScore.get_total_time():.4f} seconds")

Learn causal graph using PC algorithm...


Depth=3, working on node 4: 100%|██████████| 5/5 [00:00<00:00, 53.16it/s]  


* PC algorithm SLA elapsed CI-test time: 1.9028 seconds

Learn causal graph using FCI algorithm...


Depth=0, working on node 4: 100%|██████████| 5/5 [00:00<00:00,  9.63it/s]


X1 --> X2
X1 --> X3
X2 --> X3
* FCI algorithm SLA elapsed CI-test time: 2.3178 seconds

Learn causal graph using GES algorithm...
* GES algorithm SLA elapsed score-computation time: 1.0696 seconds
